# Reproduce the paper's headline numbers

Every headline number in the FAccT paper is verified here against the
committed artifact that backs it (the paper's Appendix G maps claim to
artifact to command; this notebook is the executable version of that map).
Where a number is recomputable from counts alone, it is recomputed from
scratch: Wilson intervals, disparate-impact ratios, four-fifths flags, and
attrition shares are re-derived here, not read back.

**What this notebook deliberately does not do: retrain models.**
`hmda_dataset/metrics/out_of_fold_metrics.json` is the frozen scoring
artifact from the source machine (Python 3.11.14, scikit-learn 1.8.0,
numpy 2.4.4, pandas 3.0.2, macOS/arm64). Re-running
`train_models.py --full-population` in a fresh environment yields
different predictions (acc 0.7982 / AUC 0.8025 observed) even at pinned
versions and seed 42, which silently changes every downstream table.
Reproducibility for that layer means verifying against the committed
file, and this notebook enforces it.

Run from the repo root or from `notebooks/`:

```
jupyter nbconvert --to notebook --execute notebooks/reproduce_facct_numbers.ipynb
```


In [1]:
import json, math, re
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "artifacts").exists():          # running from notebooks/
    ROOT = ROOT.parent
assert (ROOT / "artifacts").exists(), f"repo root not found from {Path.cwd()}"

RESULTS = []

def check(name, claimed, actual, tol=0.0):
    ok = (abs(claimed - actual) <= tol) if tol else (claimed == actual)
    RESULTS.append((name, claimed, actual, ok))
    print(f"{'PASS' if ok else 'FAIL'}  {name}: paper={claimed}  artifact={actual}")
    return ok

def load(rel):
    with (ROOT / rel).open() as f:
        return json.load(f)


## 1. Out-of-fold scoring layer (frozen artifact guard)

The paper's HMDA tables all descend from one out-of-fold scoring pass:
random forest, 5-fold, seed 42, over all 10,978 rows of the frozen
analytic sample.


In [2]:
oof = load("hmda_dataset/metrics/out_of_fold_metrics.json")
rf = oof["models"]["random_forest"]
check("OOF n_rows", 10978, oof["n_rows"])
check("OOF n_folds", 5, oof["n_folds"])
check("OOF seed", 42, oof["seed"])
check("OOF RF accuracy", 0.8002, rf["accuracy"])
check("OOF RF AUC", 0.8018, rf["auc"])


PASS  OOF n_rows: paper=10978  artifact=10978
PASS  OOF n_folds: paper=5  artifact=5
PASS  OOF seed: paper=42  artifact=42
PASS  OOF RF accuracy: paper=0.8002  artifact=0.8002
PASS  OOF RF AUC: paper=0.8018  artifact=0.8018


True

## 2. Race x sex cells: recompute Wilson intervals and DI from counts

For each of the 12 cells the artifact stores `n` and `selection_rate`.
Everything else is re-derived here: the Wilson 95% interval on each
cell's selection rate, the DI ratio against the reference cell, and the
conservative DI interval (cell Wilson bound over the opposite reference
Wilson bound). Recomputed values must match the artifact to 0.002
(rounding slack: rates are stored at 4 decimals).


In [3]:
Z = 1.959963984540054

def wilson(k, n):
    p = k / n
    d = 1 + Z * Z / n
    c = p + Z * Z / (2 * n)
    h = Z * math.sqrt(p * (1 - p) / n + Z * Z / (4 * n * n))
    return (c - h) / d, (c + h) / d

ie = load("artifacts/consolidated/interval_estimates.json")
hm = ie["hmda"]
cells_ = hm["cells"]
ref = next(c for c in cells_ if c["cell"] == hm["reference_cell"])
rk = round(ref["selection_rate"] * ref["n"])
rlo, rhi = wilson(rk, ref["n"])

check("HMDA reference cell is Asian x Female", 1, int(hm["reference_cell"] == "Asian x Female"))
check("HMDA reference n", 254, ref["n"])

worst = min(cells_, key=lambda c: c["di"])
check("worst cell DI (American Indian x Male)", 0.6734, worst["di"])

n_bad = 0
for c in cells_:
    k = round(c["selection_rate"] * c["n"])
    lo, hi = wilson(k, c["n"])
    di = c["selection_rate"] / ref["selection_rate"]
    ok = (abs(di - c["di"]) <= 0.002
          and abs(lo / rhi - c["di_ci_low"]) <= 0.002
          and abs(hi / rlo - c["di_ci_high"]) <= 0.002)
    n_bad += not ok
    print(f"{'PASS' if ok else 'FAIL'}  {c['cell']:28s} n={c['n']:5d} "
          f"DI {di:.4f} (stored {c['di']:.4f}) "
          f"CI [{lo/rhi:.4f},{hi/rlo:.4f}] (stored [{c['di_ci_low']:.4f},{c['di_ci_high']:.4f}])")
RESULTS.append(("all 12 DI ratios + CIs recompute from counts", 0, n_bad, n_bad == 0))

indeterminate = sum(not c["band_determinate"] for c in cells_)
check("indeterminate cells (interval spans 0.8)", 6, indeterminate)
check("total cells", 12, hm["n_cells"])


PASS  HMDA reference cell is Asian x Female: paper=1  artifact=1
PASS  HMDA reference n: paper=254  artifact=254
PASS  worst cell DI (American Indian x Male): paper=0.6734  artifact=0.6734
PASS  American Indian x Male       n=   31 DI 0.6734 (stored 0.6734) CI [0.4533,0.9036] (stored [0.4533,0.9036])
PASS  Multiracial x Female         n=   18 DI 0.7088 (stored 0.7088) CI [0.4295,0.9786] (stored [0.4295,0.9786])
PASS  Multiracial x Male           n=   29 DI 0.7599 (stored 0.7599) CI [0.5265,0.9831] (stored [0.5265,0.9831])
PASS  Pacific Islander x Female    n=   12 DI 0.7733 (stored 0.7732) CI [0.4344,1.0583] (stored [0.4344,1.0583])
PASS  Pacific Islander x Male      n=   15 DI 0.8505 (stored 0.8505) CI [0.5343,1.0941] (stored [0.5343,1.0941])
PASS  Black x Female               n= 1994 DI 0.8631 (stored 0.8632) CI [0.8058,0.9368] (stored [0.8058,0.9368])
PASS  Black x Male                 n= 1707 DI 0.8643 (stored 0.8643) CI [0.8051,0.9397] (stored [0.8051,0.9397])
PASS  American India

True

## 3. The reference-convention shift

Same predictions, same cells; only the choice of reference group
changes. Against the highest-rate eligible cell (Asian x Female) the two
headline cells sit at 0.6734 and 0.8632; against White x Male they move
to 0.7012 and 0.8988.


In [4]:
conv = hm["conventions"]
mrf = {c["cell"]: c["di"] for c in conv["max_rate_floor"]["cells"]}
ctl = {c["cell"]: c["di"] for c in conv["control"]["cells"]}
check("control reference is White x Male", 1, int(conv["control"]["reference_cell"] == "White x Male"))
check("AI x Male DI, max-rate reference", 0.6734, mrf["American Indian x Male"])
check("AI x Male DI, White x Male reference", 0.7012, ctl["American Indian x Male"])
check("Black x Female DI, max-rate reference", 0.8632, mrf["Black x Female"])
check("Black x Female DI, White x Male reference", 0.8988, ctl["Black x Female"])


PASS  control reference is White x Male: paper=1  artifact=1
PASS  AI x Male DI, max-rate reference: paper=0.6734  artifact=0.6734
PASS  AI x Male DI, White x Male reference: paper=0.7012  artifact=0.7012
PASS  Black x Female DI, max-rate reference: paper=0.8632  artifact=0.8632
PASS  Black x Female DI, White x Male reference: paper=0.8988  artifact=0.8988


True

## 4. Marginal DI and the four-fifths screen

The banded age attribute clears the four-fifths screen where the
ECOA-derived age-62+ cut also clears it but sits 0.09 lower; every
marginal attribute clears 0.8, which is the paper's point about what
the marginal view hides.


In [5]:
fs = load("artifacts/hmda/fairness/fairness_summary.json")
check("marginal DI, banded age_group", 0.9791, fs["age_group"]["DisparateImpact"])
check("marginal DI, age_62_plus (ECOA cut)", 0.8897, fs["age_62_plus"]["DisparateImpact"])
check("marginal DI, race", 0.9371, fs["race"]["DisparateImpact"])
check("marginal DI, sex", 0.9572, fs["sex"]["DisparateImpact"])
all_clear = all(v["DisparateImpact"] >= 0.8 for v in fs.values())
check("every marginal attribute clears the 4/5 screen", 1, int(all_clear))
worst_cell_below = worst["di"] < 0.8
check("while the worst race x sex cell fails it", 1, int(worst_cell_below))


PASS  marginal DI, banded age_group: paper=0.9791  artifact=0.9791
PASS  marginal DI, age_62_plus (ECOA cut): paper=0.8897  artifact=0.8897
PASS  marginal DI, race: paper=0.9371  artifact=0.9371
PASS  marginal DI, sex: paper=0.9572  artifact=0.9572
PASS  every marginal attribute clears the 4/5 screen: paper=1  artifact=1
PASS  while the worst race x sex cell fails it: paper=1  artifact=1


True

## 5. Subsampling study: what each small-n policy does at audit scale

2,000 draws per size, primary convention max-rate reference with a
n>=15 floor, m = 2,196 (the paper's original held-out audit size).


In [6]:
ss = load("artifacts/consolidated/subsample_study.json")
check("draws per size", 2000, ss["draws_per_size"])
check("population n", 10978, ss["population_n"])
r = ss["results"]["max_rate_floor15"]
m = "2196"
check("point estimates: false clearance", 0.8375, r["point"][m]["false_clearance"]["rate"])
check("Wilson: abstains from naming worst cell", 0.903, r["wilson"][m]["worst_cell_abstain"]["rate"])
check("Wilson: false clearance", 0.0, r["wilson"][m]["false_clearance"]["rate"])
check("EB shrinkage: false clearance", 1.0, r["eb"][m]["false_clearance"]["rate"])
check("floor n>=15: wrong worst cell", 0.9995, r["floor15"][m]["worst_cell_wrong"]["rate"])
check("floor n>=30: wrong worst cell", 1.0, r["floor30"][m]["worst_cell_wrong"]["rate"])


PASS  draws per size: paper=2000  artifact=2000
PASS  population n: paper=10978  artifact=10978
PASS  point estimates: false clearance: paper=0.8375  artifact=0.8375
PASS  Wilson: abstains from naming worst cell: paper=0.903  artifact=0.903
PASS  Wilson: false clearance: paper=0.0  artifact=0.0
PASS  EB shrinkage: false clearance: paper=1.0  artifact=1.0
PASS  floor n>=15: wrong worst cell: paper=0.9995  artifact=0.9995
PASS  floor n>=30: wrong worst cell: paper=1.0  artifact=1.0


True

## 6. Population construction

20,000 raw rows to 10,978 analytic rows; the largest single drop is
Race Not Available, 4,140 rows, 20.7% of the raw file.


In [7]:
at = load("artifacts/consolidated/population_attrition.json")
check("raw rows", 20000, at["raw_rows"])
check("final rows", 10978, at["final_rows"])
race_step = next(s for s in at["attrition"] if s["step"] == "race")
rna = race_step["dropped_by_category"]["Race Not Available"]
check("Race Not Available rows dropped", 4140, rna)
check("Race Not Available share of raw file", 0.207, rna / at["raw_rows"], tol=0.0005)


PASS  raw rows: paper=20000  artifact=20000
PASS  final rows: paper=10978  artifact=10978
PASS  Race Not Available rows dropped: paper=4140  artifact=4140
PASS  Race Not Available share of raw file: paper=0.207  artifact=0.207


True

## 7. Layer one: the library-default reproduction

`scripts/aif360_default_repro.py` at pinned versions: the constructor
default reads DI 1.2072 on a column whose favorable outcome is 0, the
oriented call reads 0.9862, and Fairlearn reads 0.8284 on the column as
encoded.


In [8]:
txt = (ROOT / "artifacts/consolidated/aif360_default_repro.txt").read_text()
nums = [float(x) for x in re.findall(r"DI = ([0-9.]+)", txt)]
check("AIF360 default favorable_label DI", 1.2072, nums[0])
check("AIF360 oriented DI", 0.9862, nums[1])
fl = float(re.search(r"column as encoded:\n\s+([0-9.]+)", txt).group(1))
check("Fairlearn DP ratio, column as encoded", 0.8284, fl)


PASS  AIF360 default favorable_label DI: paper=1.2072  artifact=1.2072
PASS  AIF360 oriented DI: paper=0.9862  artifact=0.9862
PASS  Fairlearn DP ratio, column as encoded: paper=0.8284  artifact=0.8284


True

## 8. German Credit cross-check


In [9]:
gc = {c["cell"]: c["di"] for c in ie["german_credit"]["cells"]}
check("German Credit under-40 female DI", 0.8201, gc["under_40 x female"])


PASS  German Credit under-40 female DI: paper=0.8201  artifact=0.8201


True

## Verdict


In [10]:
n_fail = sum(not ok for *_, ok in RESULTS)
print(f"{len(RESULTS)} checks, {n_fail} failures")
for name, claimed, actual, ok in RESULTS:
    if not ok:
        print(f"  FAIL {name}: paper={claimed} artifact={actual}")
assert n_fail == 0, f"{n_fail} paper numbers do not match their artifacts"
print("Every checked number in the paper matches its committed artifact.")


38 checks, 0 failures
Every checked number in the paper matches its committed artifact.
